# Transformer Explained

This notebook explains the core idea behind the Transformer and shows a tiny, runnable attention example.

The core question is:

- Traditional RNNs process tokens one by one.
- Transformers let every token look at every other token in parallel.
- This is done with self-attention.

The main operation is:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

where:
- $Q$ = queries
- $K$ = keys
- $V$ = values
- $d_k$ = dimension of keys


## 1. Why attention?

In a sentence, each word matters differently depending on context:

> "The animal didn't cross the street because it was too tired."

The word `it` could refer to the animal or the street depending on context.

A Transformer does not keep a single hidden state like RNNs. Instead, it computes attention weights for each token as it relates to every other token.

This makes it much easier to model long-range dependencies.

## 2. Key idea: Q, K, V

Each token is transformed into three vectors:

- Query $Q$: what information am I looking for?
- Key $K$: what information do I contain?
- Value $V$: what content should be passed along if selected?

For each token, we compute:

$$
score_{ij} = \frac{Q_i \cdot K_j}{\sqrt{d_k}}
$$

Then softmax turns the scores into attention weights.

Finally:

$$
output_i = \sum_j \alpha_{ij} V_j
$$

This means each token updates itself by gathering information from relevant tokens.

In [4]:
import torch

# Tiny toy example: 3 tokens, each token has 2 features
x = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]], dtype=torch.float32)

# Simple learned projection matrices
W_q = torch.tensor([[1.0, 0.0],
                   [0.5, 0.8]], dtype=torch.float32)
W_k = torch.tensor([[1.0, 0.1],
                   [0.2, 0.9]], dtype=torch.float32)
W_v = torch.tensor([[1.0, 0.0],
                   [0.0, 1.0]], dtype=torch.float32)

Q = x @ W_q
K = x @ W_k
V = x @ W_v

scores = Q @ K.T / (K.shape[1] ** 0.5)
weights = torch.softmax(scores, dim=-1)
output = weights @ V

print('x:\n', x)
print('\nQ:\n', Q)
print('\nK:\n', K)
print('\nV:\n', V)
print('\nAttention weights:\n', weights)
print('\nOutput:\n', output)


x:
 tensor([[1., 0.],
        [0., 1.],
        [1., 1.]])

Q:
 tensor([[1.0000, 0.0000],
        [0.5000, 0.8000],
        [1.5000, 0.8000]])

K:
 tensor([[1.0000, 0.1000],
        [0.2000, 0.9000],
        [1.2000, 1.0000]])

V:
 tensor([[1., 0.],
        [0., 1.],
        [1., 1.]])

Attention weights:
 tensor([[0.3677, 0.2088, 0.4235],
        [0.2518, 0.2984, 0.4497],
        [0.2681, 0.1804, 0.5515]])

Output:
 tensor([[0.7912, 0.6323],
        [0.7016, 0.7482],
        [0.8196, 0.7319]])


## 3. Why divide by sqrt(d_k)?

The dot products can become too large as the feature dimension grows. This makes softmax saturate and produce nearly one-hot distributions.

So we scale by $\sqrt{d_k}$:

$$
score = \frac{QK^T}{\sqrt{d_k}}
$$

This keeps the scale stable and helps learning remain smooth.

## 4. Multi-head attention

Instead of using one attention pattern, the Transformer uses multiple heads.

Each head learns a different way to attend. Then the results are concatenated and projected back to the model size.

This lets the model capture:
- syntax
- coreference
- semantic relationships
- positional structure

A typical multi-head attention block is:

1. Linear projections to $Q$, $K$, $V$
2. Split into heads
3. Compute attention per head
4. Concatenate heads
5. Final projection

## 5. Positional encoding

Self-attention itself is permutation-invariant. That means if you shuffle token order, the model sees the same bag of words unless you add position information.

The original Transformer uses sine and cosine positional encodings:

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)
$$

These are added to token embeddings so the model knows the order of words.

In [ ]:
import math

import torch


def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    for pos in range(seq_len):
        for i in range(d_model):
            angle = pos / (10000 ** ((2 * (i // 2)) / d_model))
            if i % 2 == 0:
                pe[pos, i] = math.sin(angle)
            else:
                pe[pos, i] = math.cos(angle)
    return pe


pe = positional_encoding(5, 8)
print(pe)


tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9996e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00],
        [-7.5680e-01, -6.5364e-01,  3.8942e-01,  9.2106e-01,  3.9989e-02,
          9.9920e-01,  4.0000e-03,  9.9999e-01]])


## 6. Transformer block structure

A standard Transformer block contains:

- Multi-head self-attention
- Residual connection
- Layer normalization
- Feed-forward network
- Residual connection
- Layer normalization

The core block is roughly:

$$
x = x + \mathrm{MHA}(\mathrm{LN}(x))
$$

$$
x = x + \mathrm{FFN}(\mathrm{LN}(x))
$$

This residual design makes training deep networks much easier.

## 7. Why Transformers work so well

Transformers are powerful because they:

- process many tokens in parallel
- directly model token-to-token relationships
- avoid the sequential bottleneck of RNNs
- scale well to large datasets and large models

This made them the foundation for modern large language models (LLMs).

## 8. Summary

The Transformer is built around attention:

- tokens become Q, K, V
- attention scores compare each query to keys
- softmax produces weights
- values are weighted and aggregated
- multiple heads and layers build rich representations

This is why Transformers dominate NLP, vision, and multimodal modeling today.